# Vietnam — sneaker classifier (Colab)

## How to use: Runtime → Run all. Nothing to change.
* **If `MyDrive/vietnam/output_simple/model.pt` exists** (your case): the notebook keeps that model and only adjusts its
  threshold `tau` (takes seconds, no GPU needed). It writes **`output_simple/SUBMIT_THIS/model.pt`** → submit that file.
  Your original `model.pt` is not changed.
* **If there is no `model.pt`** (or `FORCE_TRAIN = True`): it trains a new model (~15 min, needs a T4 GPU:
  Runtime → Change runtime type → T4 GPU) and saves it in `output_simple/run_<date>_<time>/`.

## Why only tau
The model gives 3 probabilities (nike, adidas, converse). The evaluator answers the top brand, or **`unknown` if the top
probability is below `tau`**. With tau 0.95 the model got known **76/90** (need 77) and rejection **146/180**
(need only 126). Lowering tau to **0.92** gave a **PASS** on both.
So tau is slightly too strict: a few real shoes are called unknown. A slightly **lower tau** lets them through and gives
away a few of the 20 spare rejections.

If the result after submitting is still a FAIL:
* known accuracy still < 0.85 → submit `output_simple/backup_lower_tau/model.pt`
* rejection < 0.70 → submit `output_simple/backup_higher_tau/model.pt`

## Model (for reference)
Pretrained ConvNeXt-Tiny, 3 classes; blocks not in `KEEP_BLOCKS` are pass-through so the fp16 file is ≈ 8.5 MB (limit 10 MB),
with the exact 182 keys/shapes of `torchvision convnext_tiny(num_classes=3)` and classes `['nike', 'adidas', 'converse']`.
Trained with fake unknowns (jigsaw / blend / half-swap of the training photos) pushed toward equal ⅓ probabilities.

## 1. Settings

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/vietnam'   # folder with vietnam_data.zip
ZIP_NAME  = 'vietnam_data.zip'

FORCE_TRAIN = False    # False: if output_simple/model.pt exists, only adjust its tau. True: train a new model.
TAU_STEP_SUBMIT  = 0.03            # SUBMIT_THIS tau  = model.pt tau - 0.03
TAU_STEP_LOWER   = 0.05            # backup_lower_tau  (more known accepted, fewer rejected)
TAU_STEP_HIGHER  = 0.015           # backup_higher_tau (fewer known accepted, more rejected)

# ---- training settings (only used when training) ----

RUN_FULL     = True    # after run A picks tau, retrain on all 735 images (run B) and save that. False: save run A's model.
TARGET_KNOWN = 0.90    # model.pt tau = highest value whose validation known-accuracy is still >= this (evaluator gate: 0.85)
TAU_OVERRIDE = None    # set e.g. 0.97 to force model.pt's tau instead of the automatic choice
# extra files with the SAME weights, only tau differs. Higher tau -> more 'unknown' (rejection up, known accuracy down).
TAU_SWEEP    = [0.950, 0.955, 0.960, 0.965, 0.970, 0.975, 0.980, 0.985, 0.990, 0.995]

CLASSES   = ['nike', 'adidas', 'converse']   # order required by the evaluator - do not sort
SIZE      = 224
EPOCHS    = 25
WARMUP    = 2          # epochs of linear LR warm-up
BATCH     = 32
LR_BODY   = 1e-4       # pretrained layers
LR_HEAD   = 1e-3       # new 3-class layer
WEIGHT_DECAY = 0.05
EMA_DECAY = 0.995      # the saved/evaluated model is the moving average of the weights
NEG_FRAC  = 0.5        # fake unknowns per real image
OE_WEIGHT = 0.5        # weight of the 'be unsure on fake unknowns' loss (higher = more rejection, lower known accuracy)
SEED      = 0
MAX_MB    = 10.0

# blocks with real weights (all others are pass-through). fp16 sizes:
#   features.1.x 0.16 MB, features.3.x 0.61 MB, features.5.x 2.40 MB, features.7.x 9.53 MB (never fits)
#   always stored: stem + downsamples + head = 1.35 MB with TIE_DOWNSAMPLE (3.1 MB without)
KEEP_BLOCKS = ['features.1.0', 'features.1.1', 'features.1.2',
               'features.3.0', 'features.3.1', 'features.3.2',
               'features.5.0', 'features.5.8']                 # -> about 8.5 MB
TIE_DOWNSAMPLE = True

SMOKE_TEST = False     # True = tiny CPU run that only checks the code works

## 2. Mount Drive

In [ ]:
import os, subprocess, time
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    OUT_DIR = f'{DRIVE_DIR}/output_simple'
    os.chdir('/content')
else:
    OUT_DIR = 'output'          # local run: expects data/ next to the notebook
os.makedirs(OUT_DIR, exist_ok=True)
EXISTING = f'{OUT_DIR}/model.pt'
MODE = 'train' if FORCE_TRAIN or not os.path.exists(EXISTING) else 'fix'
if MODE == 'train' and IN_COLAB and not os.path.exists('data/train-input'):
    subprocess.run(['unzip', '-q', '-o', f'{DRIVE_DIR}/{ZIP_NAME}', '-d', '/content'], check=True)
print('files in output_simple:', sorted(os.listdir(OUT_DIR)))
print('MODE:', 'fix tau of the existing model.pt' if MODE == 'fix' else 'train a new model')

## 3. Fix tau of the existing model.pt → `SUBMIT_THIS/model.pt` (skipped when training)

In [ ]:
import torch, torchvision.models as tvm

class StopHere(Exception):            # ends "Run all" quietly - the training cells below are not needed
    def _render_traceback_(self): return []

if MODE == 'fix':
    ck = torch.load(EXISTING, map_location='cpu', weights_only=True)
    assert list(ck) == ['classes', 'tau', 'mean', 'std', 'state_dict'], list(ck)
    m = tvm.convnext_tiny(weights=None, num_classes=3)          # same check the evaluator does
    m.load_state_dict({k: v.float() for k, v in ck['state_dict'].items()}, strict=True)
    old = float(ck['tau'])
    print(f'output_simple/model.pt: tau = {old}')
    if old >= 0.80:
        plan = {'SUBMIT_THIS': old - TAU_STEP_SUBMIT,
                'backup_lower_tau': old - TAU_STEP_LOWER,
                'backup_higher_tau': old - TAU_STEP_HIGHER}
    else:   # tau 0.5 model: almost nothing was rejected (0.16) -> tau must go UP a lot
        print('this model has a LOW tau (it rejected almost nothing before) -> raising tau instead')
        plan = {'SUBMIT_THIS': 0.97, 'backup_lower_tau': 0.95, 'backup_higher_tau': 0.985}
    for folder, tau in plan.items():
        os.makedirs(f'{OUT_DIR}/{folder}', exist_ok=True)
        ck['tau'] = round(tau, 4)
        dst = f'{OUT_DIR}/{folder}/model.pt'
        torch.save(ck, dst)                                    # same weights, only tau changes
        mb = os.path.getsize(dst) / 1e6
        assert mb <= MAX_MB, mb
        print(f'  {folder + "/model.pt":28s} tau {ck["tau"]:.4f}   ({mb:.2f} MB)')
    print(f'\n>>> SUBMIT:  MyDrive/vietnam/output_simple/SUBMIT_THIS/model.pt   (tau {plan["SUBMIT_THIS"]:.4f})')
    print('    still FAIL on known accuracy -> submit backup_lower_tau/model.pt')
    print('    still FAIL on rejection      -> submit backup_higher_tau/model.pt')
    if IN_COLAB:
        from google.colab import files
        files.download(f'{OUT_DIR}/SUBMIT_THIS/model.pt')
    raise StopHere()
print('training a new model - continuing with the cells below')

## 4. Data and model (training only)

In [ ]:
import math, random, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import torchvision.models as tvm, torchvision.transforms as T
from torch.utils.data import DataLoader, Dataset
from PIL import Image

def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s)
seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
AMP = DEVICE == 'cuda'
print('device:', DEVICE, '| torch', torch.__version__)
if DEVICE == 'cpu' and not SMOKE_TEST:
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then run all again.')
if SMOKE_TEST:
    EPOCHS, WARMUP = 1, 0

IMG_DIR, CSV = 'data/train-input', 'data/train-output.csv'
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([T.RandomResizedCrop(SIZE, scale=(0.6, 1.0)), T.RandomHorizontalFlip(),
                      T.RandomApply([T.TrivialAugmentWide()], p=0.5),
                      T.ToTensor(), T.Normalize(MEAN, STD), T.RandomErasing(p=0.2, scale=(0.02, 0.15))])
eval_tf  = T.Compose([T.Resize((SIZE, SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

df = pd.read_csv(CSV)
assert set(df.label) == set(CLASSES), set(df.label)
if SMOKE_TEST:
    df = df.groupby('label').head(20).reset_index(drop=True)
df['y'] = df.label.map({c: i for i, c in enumerate(CLASSES)})
IMGS = {f: Image.open(f'{IMG_DIR}/{f}').convert('RGB') for f in df.filename}   # decoded once, kept in RAM
def load(f): return IMGS[f]

class Shoes(Dataset):
    def __init__(self, frame, tf):
        self.files, self.y, self.tf = frame.filename.tolist(), frame.y.tolist(), tf
    def __len__(self): return len(self.files)
    def __getitem__(self, i): return self.tf(load(self.files[i])), self.y[i]

# ---- model ----
def is_dropped(key):
    # True for parameters of a ConvNeXt block that is not in KEEP_BLOCKS
    p = key.split('.')
    return p[0] == 'features' and p[1] in ('1', '3', '5', '7') and '.'.join(p[:3]) not in KEEP_BLOCKS

class TiedConv2x2(nn.Module):
    # 2x2 stride-2 conv whose 4 kernel positions share one weight. Same state_dict keys as nn.Conv2d,
    # weight kept as [out, in, 1, 1] and expanded to [out, in, 2, 2] when saved -> 1/4 of the file size.
    def __init__(self, conv):
        super().__init__()
        self.weight = nn.Parameter(conv.weight.detach().mean((2, 3), keepdim=True).clone())
        self.bias = nn.Parameter(conv.bias.detach().clone())
    def forward(self, x):
        return F.conv2d(x, self.weight.expand(-1, -1, 2, 2), self.bias, stride=2)

def make_model(n):
    m = tvm.convnext_tiny(weights='DEFAULT')
    m.classifier[2] = nn.Linear(m.classifier[2].in_features, n)
    if TIE_DOWNSAMPLE:
        m.features[6][1] = TiedConv2x2(m.features[6][1])
    with torch.no_grad():                 # dropped blocks: zero weights + layer_scale 0 = pass-through, frozen
        for k, p in m.named_parameters():
            if is_dropped(k):
                p.zero_(); p.requires_grad_(False)
    return m

# ---- fake unknowns, built only from the given images ----
def jigsaw(img, g=4):
    a = np.asarray(img); h, w = a.shape[0] // g * g, a.shape[1] // g * g
    tiles = [a[r*h//g:(r+1)*h//g, c*w//g:(c+1)*w//g] for r in range(g) for c in range(g)]
    random.shuffle(tiles)
    return Image.fromarray(np.concatenate([np.concatenate(tiles[r*g:(r+1)*g], 1) for r in range(g)], 0))

def blend(a, b): return Image.blend(a, b.resize(a.size), 0.5)

def half_swap(a, b):
    a, b = np.asarray(a).copy(), np.asarray(b.resize(a.size))
    if random.random() < 0.5: a[:, a.shape[1] // 2:] = b[:, a.shape[1] // 2:]
    else:                     a[a.shape[0] // 2:] = b[a.shape[0] // 2:]
    return Image.fromarray(a)

def make_negative(frame):
    ca, cb = random.sample(sorted(frame.label.unique()), 2)          # two different brands
    a = load(frame[frame.label == ca].filename.sample(1).item())
    b = load(frame[frame.label == cb].filename.sample(1).item())
    kind = random.choice(['jigsaw', 'blend', 'half_swap'])
    return jigsaw(a) if kind == 'jigsaw' else blend(a, b) if kind == 'blend' else half_swap(a, b)

class TrainSet(Dataset):
    # real images with their label + NEG_FRAC * as many fake unknowns with label -1 (made fresh every time)
    def __init__(self, frame):
        self.f = frame.reset_index(drop=True)
        self.n_pos, self.n_neg = len(self.f), int(len(self.f) * NEG_FRAC)
    def __len__(self): return self.n_pos + self.n_neg
    def __getitem__(self, i):
        if i < self.n_pos:
            return train_tf(load(self.f.filename[i])), int(self.f.y[i])
        return train_tf(make_negative(self.f)), -1

class ImgList(Dataset):
    def __init__(self, imgs): self.imgs = imgs
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i): return eval_tf(self.imgs[i]), 0

@torch.no_grad()
def _predict(model, ds):
    model.eval(); out = []
    for x, _ in DataLoader(ds, batch_size=64, num_workers=2):
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
            out.append(model(x.to(DEVICE)).float().softmax(1).cpu())
    return torch.cat(out).numpy()

def predict_probs(model, frame): return _predict(model, Shoes(frame, eval_tf))
def predict_images(model, imgs): return _predict(model, ImgList(imgs))

# ---- split: run A trains on 80%, the other 20% picks tau ----
val = df.groupby('label').sample(frac=0.2, random_state=SEED)
train_part = df.drop(val.index)
random.seed(SEED + 1)
val_neg = [make_negative(val) for _ in range(150)]    # fixed fake unknowns made ONLY from validation images
seed_all(SEED)
print(f'classes {CLASSES} | all {len(df)} | run A: train {len(train_part)}, val {len(val)} + {len(val_neg)} fake unknowns')

## 5. Training function

In [ ]:
def total_loss(logits, y):
    known = y >= 0
    loss = F.cross_entropy(logits[known], y[known]) if known.any() else logits.sum() * 0
    if (~known).any():   # fake unknowns: cross-entropy to the uniform (1/3, 1/3, 1/3) distribution
        loss = loss + OE_WEIGHT * (-logits[~known].log_softmax(1).mean(1)).mean()
    return loss

def val_report(model, tau):
    p = predict_probs(model, val)
    ok, conf = p.argmax(1) == val.y.to_numpy(), p.max(1)
    rej = (predict_images(model, val_neg).max(1) < tau).mean()
    return f'val acc {ok.mean():.3f} | known acc @{tau:.2f} {(ok & (conf >= tau)).mean():.3f} | fake-unknown rejection @{tau:.2f} {rej:.3f}'

def train_model(frame, show_val):
    seed_all(SEED)
    model = make_model(len(CLASSES)).to(DEVICE)
    body = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('classifier')]
    head = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith('classifier')]
    print(f'trainable params: {sum(p.numel() for p in body + head) / 1e6:.1f} M')
    opt = torch.optim.AdamW([{'params': body, 'lr': LR_BODY}, {'params': head, 'lr': LR_HEAD}], weight_decay=WEIGHT_DECAY)
    dl = DataLoader(TrainSet(frame), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True, persistent_workers=True)
    total, warm = EPOCHS * len(dl), WARMUP * len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    ema = torch.optim.swa_utils.AveragedModel(
        model, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(EMA_DECAY), use_buffers=True)
    scaler = torch.amp.GradScaler(enabled=AMP)
    for ep in range(EPOCHS):
        model.train(); tot = n = 0
        for x, y in dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
                logits = model(x)
            loss = total_loss(logits.float(), y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            ema.update_parameters(model)
            tot += loss.item() * len(y); n += len(y)
        msg = f'epoch {ep + 1:2d}/{EPOCHS}  loss {tot / n:.4f}'
        if show_val and (ep % 5 == 4 or ep == EPOCHS - 1):
            msg += '  | EMA ' + val_report(ema.module, 0.8)
        print(msg, flush=True)
    return ema.module

## 6. Run A: train on 80%, then pick tau on the held-out 20%

In [ ]:
model_a = train_model(train_part, show_val=True)

In [ ]:
p = predict_probs(model_a, val)
ok, conf = p.argmax(1) == val.y.to_numpy(), p.max(1)
neg_conf = predict_images(model_a, val_neg).max(1)
grid = np.round(np.r_[np.arange(0.50, 0.95, 0.01), np.arange(0.95, 0.999, 0.001)], 3)
known = np.array([(ok & (conf >= t)).mean() for t in grid])
rej = np.array([(neg_conf < t).mean() for t in grid])

print(f'val accuracy without threshold: {ok.mean():.3f}\n')
print(' tau | known acc (gate 0.85) | fake-unknown rejection (gate 0.70)')
for t, k, r in zip(grid, known, rej):
    if round(t * 1000) % 50 == 0 or np.isclose(TAU_SWEEP, t).any():
        print(f'{t:.3f} |        {k:.3f}          |        {r:.3f}')

good = grid[known >= TARGET_KNOWN]
if TAU_OVERRIDE is not None:
    TAU = float(TAU_OVERRIDE)
elif len(good):
    TAU = float(good.max())
else:   # run A (80% data) is weaker than run B; never fall back to a low tau, that rejects almost nothing
    TAU = 0.95
    print(f'\nWARNING: no tau reaches val known acc {TARGET_KNOWN} in run A; using {TAU}. Try the model_tau_*.pt files.')
i = int(np.abs(grid - TAU).argmin())
print(f'\nmodel.pt tau = {TAU:.3f}  ->  val known acc {known[i]:.3f}, fake-unknown rejection {rej[i]:.3f}')
print('(real other-brand shoes are harder to reject than fake unknowns; v1 at tau 0.90 had real rejection 0.89)')

## 7. Run B: retrain on all images (same settings) for the submission

In [ ]:
if RUN_FULL:
    final_model = train_model(df, show_val=False)
else:
    final_model = model_a

## 8. Save `model.pt` (evaluator format, fp16, ≤ 10 MB) to Drive

In [ ]:
def export_state_dict(model):
    sd = {}
    for k, v in model.state_dict().items():
        v = v.detach().cpu()
        if is_dropped(k):                                   # one stored number, expanded (stride 0) to the full shape
            sd[k] = torch.zeros(1, dtype=torch.float16).expand(v.shape)
        elif TIE_DOWNSAMPLE and k == 'features.6.1.weight': # [768, 384, 1, 1] stored, seen as [768, 384, 2, 2]
            sd[k] = v.half().clone().expand(-1, -1, 2, 2)
        else:
            sd[k] = v.half().clone()
    return sd

SD = export_state_dict(final_model)
def save(path, tau):
    torch.save(dict(classes=list(CLASSES), tau=float(tau), mean=MEAN, std=STD, state_dict=SD), path)
    mb = os.path.getsize(path) / 1e6
    assert mb <= MAX_MB, f'{path} is {mb:.2f} MB > {MAX_MB} MB - remove a block from KEEP_BLOCKS'
    print(f'saved {path}  ({mb:.2f} MB, tau {tau})')

RUN_DIR = f'{OUT_DIR}/run_{time.strftime("%m%d_%H%M")}'    # new folder per run: older models are never overwritten
os.makedirs(RUN_DIR, exist_ok=True)
MODEL_PT = f'{RUN_DIR}/model.pt'
save(MODEL_PT, TAU)
SWEEP_FILES = []
for t in TAU_SWEEP:
    SWEEP_FILES.append(f'{RUN_DIR}/model_tau_{t:.3f}.pt'); save(SWEEP_FILES[-1], t)
print('\nEvaluate model_tau_*.pt from low tau to high; submit the first where known >= 0.85 AND rejection >= 0.70.')
print('Rename that file to model.pt if the evaluator needs that name.')

## 9. Check: load it the way the evaluator does (plain torchvision ConvNeXt-Tiny) and compare

In [ ]:
ck = torch.load(SWEEP_FILES[0], map_location='cpu', weights_only=True)
assert list(ck) == ['classes', 'tau', 'mean', 'std', 'state_dict'] and ck['classes'] == ['nike', 'adidas', 'converse']
m = tvm.convnext_tiny(weights=None, num_classes=3)
print(m.load_state_dict({k: v.float() for k, v in ck['state_dict'].items()}, strict=True))
m = m.to(DEVICE).eval()

chk = val if not RUN_FULL else df.groupby('label').sample(10, random_state=1)
p_saved, p_trained = predict_probs(m, chk), predict_probs(final_model, chk)
print(f'max prob difference saved vs trained model: {np.abs(p_saved - p_trained).max():.4f}  (should be ~0.01 or less)')
c_all = predict_probs(m, df).max(1)
print('share of the 735 training images (seen in training, so optimistic) still >= each tau:')
print('  ' + '  '.join(f'{t}: {(c_all >= t).mean():.3f}' for t in TAU_SWEEP))
pred = [ck['classes'][i] if c >= ck['tau'] else 'unknown' for c, i in zip(p_saved.max(1), p_saved.argmax(1))]
for f, true, pr, c in list(zip(chk.filename, chk.label, pred, p_saved.max(1)))[:15]:
    print(f'{f}  true={true:9s} pred={pr:9s} conf={c:.3f}')

## 10. Download `model.pt` (optional)

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(MODEL_PT)
    for f in SWEEP_FILES: files.download(f)   # all files are also in the Drive folder RUN_DIR